Data Preparation and Feature Engineering

Vehicle Valuation and Depreciation Intelligence Platform

Objective

The objective of this notebook is to prepare the used-vehicle listing data
for a baseline multiple linear regression model that predicts current listing
price.

The preparation process applies findings from exploratory data analysis,
including duplicate records, redundant columns, missing values, implausible
mileage observations, and high-cardinality categorical features.

Model training and train/test splitting are outside the scope of this notebook.

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
DATA_PATH = Path("../data/processed/used_cars_reduced.csv")
df = pd.read_csv(DATA_PATH)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

C:\Users\casey\AppData\Local\Temp\ipykernel_28548\3415453124.py:5: DtypeWarning: Columns (0: dealer_zip) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


In [82]:
pd.set_option("display.max_columns", None)


In [83]:
"index" in df.columns

False

In [84]:
df

,city,dealer_zip,engine_cylinders,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,vehicle_damage_category,wheel_system,year
0,Bayamon,960,I4,I4,NaN,Gasoline,NaN,177.0,1,Jeep,7.0,Renegade,NaN,23141.0,NaN,2.800000,A,Latitude FWD,NaN,FWD,2019
1,San Juan,922,I4,I4,NaN,Gasoline,NaN,246.0,1,Land Rover,8.0,Discovery Sport,NaN,46500.0,NaN,3.000000,A,S AWD,NaN,AWD,2020
2,Guaynabo,969,H4,H4,False,Gasoline,False,305.0,0,Subaru,NaN,WRX STI,3.0,46995.0,False,NaN,M,Base,NaN,AWD,2016
3,San Juan,922,V6,V6,NaN,Gasoline,NaN,340.0,1,Land Rover,11.0,Discovery,NaN,67430.0,NaN,3.000000,A,V6 HSE AWD,NaN,AWD,2020
4,San Juan,922,I4,I4,NaN,Gasoline,NaN,246.0,1,Land Rover,7.0,Discovery Sport,NaN,48880.0,NaN,3.000000,A,S AWD,NaN,AWD,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663246,Fairfield,94533,I4,I4,False,Gasoline,False,170.0,0,Chevrolet,41897.0,Equinox,1.0,17998.0,False,4.272727,A,1.5T LT FWD,NaN,FWD,2018
2663247,Vallejo,94591,V6,V6,NaN,Gasoline,NaN,310.0,1,Chevrolet,5.0,Traverse,NaN,36490.0,NaN,4.533333,A,LS FWD,NaN,FWD,2020
2663248,Napa,94559,NaN,NaN,False,NaN,True,240.0,0,Ford,57992.0,Fusion,2.0,12990.0,False,4.142857,A,SE,NaN,FWD,2016
2663249,Fairfield,94533,I4 Diesel,I4 Diesel,False,Diesel,False,180.0,0,Jaguar,27857.0,XE,1.0,26998.0,False,4.272727,A,20d Premium AWD,NaN,AWD,2017


In [10]:
df.shape


(2663251, 21)

PHASE 1 i
Do new vehicles belong in the dataset?
Yes New vehicles provide the starting market reference before mileage, age, ownership, accidents, and other factors reduce value.


ii. Target Price Validation

In [12]:
df['price'].isna()

0          False
1          False
2          False
3          False
4          False
           ...  
2663246    False
2663247    False
2663248    False
2663249    False
2663250    False
Name: price, Length: 2663251, dtype: bool

In [13]:
df['price'].isna().sum()

np.int64(0)

In [21]:
(df["price"] == 0).sum()

np.int64(0)

In [22]:
(df["price"] < 0).sum()

np.int64(0)

In [33]:
df['price'].max()
df.loc[df["price"] == df["price"].max()]

,city,dealer_zip,engine_cylinders,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,vehicle_damage_category,wheel_system,year
1261170,Palm Harbor,34683,V12,V12,False,Gasoline,False,660.0,0,Ferrari,5339.0,Enzo,2.0,3299995.0,False,4.4,A,2 Dr STD Coupe,NaN,RWD,2003


In [35]:
df['price'].min()
df.loc[df["price"] == df["price"].min()]

,city,dealer_zip,engine_cylinders,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,vehicle_damage_category,wheel_system,year
1320599,Belle Glade,33430,V6,V6,False,Gasoline,False,160.0,0,Buick,190000.0,Century,4.0,165.0,False,3.565217,A,Custom Sedan FWD,NaN,FWD,1999


In [ ]:
df['mileage'] 

0              7.0
1              8.0
2              NaN
3             11.0
4              7.0
            ...   
2663246    41897.0
2663247        5.0
2663248    57992.0
2663249    27857.0
2663250    22600.0
Name: mileage, Length: 2663251, dtype: float64

In [40]:

comparison=df.loc[(df["make_name"]== 'Buick') & 
                  (df["model_name"] == "Century"),
[       
        "year",
        "mileage",
        "owner_count",
        "has_accidents",
        "frame_damaged",
        "salvage",
        "price"
   ]
]
comparison.shape
comparison.sort_values("price").head(20)

,year,mileage,owner_count,has_accidents,frame_damaged,salvage,price
1320599,1999,190000.0,4.0,False,False,False,165.0
878365,2005,202158.0,5.0,True,False,False,249.0
301631,2001,175689.0,2.0,False,False,False,495.0
1148231,2002,NaN,5.0,False,False,False,600.0
1209120,2002,150000.0,4.0,False,False,False,750.0
1687931,2003,160730.0,4.0,True,False,False,995.0
1148579,1999,130474.0,3.0,True,False,False,1400.0
1738903,2002,236760.0,4.0,True,False,False,1490.0
44862,2001,185524.0,2.0,False,False,False,1499.0
1523587,2004,165385.0,5.0,False,False,False,1500.0


In [44]:
df["frame_damaged"].any()

np.True_

In [47]:
df.sort_values("price").head(5)

,city,dealer_zip,engine_cylinders,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,vehicle_damage_category,wheel_system,year
1320599,Belle Glade,33430,V6,V6,False,Gasoline,False,160.0,0,Buick,190000.0,Century,4.0,165.0,False,3.565217,A,Custom Sedan FWD,NaN,FWD,1999
1359718,Medley,33178,NaN,NaN,False,NaN,False,210.0,0,Ford,150000.0,Explorer,4.0,200.0,False,4.000000,A,XLT V6,NaN,RWD,2005
878365,Traverse City,49684,V6,V6,False,Gasoline,True,175.0,0,Buick,202158.0,Century,5.0,249.0,False,3.593750,A,Custom Sedan FWD,NaN,FWD,2005
1271928,Bainbridge,39817,NaN,NaN,False,NaN,True,170.0,0,Nissan,NaN,Altima Coupe,5.0,250.0,False,1.000000,CVT,2.5 S,NaN,FWD,2008
1271923,Bainbridge,39817,V6,V6,False,Gasoline,False,290.0,0,Acura,NaN,RL,3.0,250.0,False,1.000000,A,SH-AWD with Navigation and Tech Package,NaN,AWD,2006


In [48]:
(df["price"] < 500).sum()

np.int64(86)

In [58]:
(df["price"] < 1000).sum()

np.int64(494)

In [ ]:
df.loc[
    df["price"] < 1000,
    [
        "make_name",
        "model_name",
        "year",
        "is_new"
        "mileage",
        "owner_count",
        "has_accidents",
        "frame_damaged",
        "salvage",
        
        "price"
    ]
].sort_values("price").head(30)

,make_name,model_name,year,mileage,owner_count,has_accidents,frame_damaged,salvage,price
1320599,Buick,Century,1999,190000.0,4.0,False,False,False,165.0
1359718,Ford,Explorer,2005,150000.0,4.0,False,False,False,200.0
878365,Buick,Century,2005,202158.0,5.0,True,False,False,249.0
1271941,Kia,Sorento,2006,NaN,6.0,True,False,False,250.0
1271923,Acura,RL,2006,NaN,3.0,False,False,False,250.0
1271928,Nissan,Altima Coupe,2008,NaN,5.0,True,False,False,250.0
1271918,Toyota,Avalon,2005,NaN,5.0,True,False,False,250.0
1271913,Mitsubishi,Lancer,2008,NaN,2.0,True,False,False,250.0
1938898,Mercury,Villager,1998,104000.0,3.0,False,False,False,256.0
1579657,Pontiac,Vibe,2003,200000.0,5.0,True,False,False,299.0


In [55]:
comparison_Nissan=df.loc[(df["make_name"]== 'Nissan') & 
                  (df["model_name"] == "Altima"),
[       
        "year",
        "mileage",
        "owner_count",
        "has_accidents",
        "frame_damaged",
        "salvage",
        "price"
   ]
]
comparison_Nissan.shape
comparison_Nissan.sort_values("price").head(20)

,year,mileage,owner_count,has_accidents,frame_damaged,salvage,price
1394034,2020,NaN,1.0,False,False,False,386.9
1320684,2004,175222.0,7.0,True,False,False,484.0
1320691,1998,177420.0,2.0,False,False,False,484.0
949965,1997,NaN,3.0,True,False,False,550.0
1579680,2006,204952.0,4.0,True,False,False,677.0
1677446,2005,200829.0,6.0,True,False,False,800.0
1250485,2003,191652.0,NaN,False,False,False,899.0
950307,2002,NaN,4.0,False,False,False,900.0
559459,2003,NaN,2.0,True,False,False,900.0
950193,1997,NaN,4.0,True,False,False,900.0


In [59]:
df.loc[df["price"] < 1000, "price"].value_counts()

price
999.0    95
995.0    64
900.0    32
484.0    27
495.0    23
         ..
389.0     1
980.0     1
256.0     1
890.0     1
449.0     1
Name: count, Length: 72, dtype: int64

In [68]:
df.loc[
    df["price"] == 999,
    [
        "make_name",
        "model_name",
        "year",
        "mileage",
        "city",
        "is_new",
        "owner_count",
        "has_accidents",
        "frame_damaged",
        "salvage",
        'dealer_zip',
        "price"
    ]
].sort_values("price").head(30)

,make_name,model_name,year,mileage,city,is_new,owner_count,has_accidents,frame_damaged,salvage,dealer_zip,price
2686,Honda,Civic,1996,222434.0,Teterboro,0,4.0,True,False,False,7608,999.0
2690,Nissan,Maxima,2001,NaN,Teterboro,0,3.0,True,False,False,7608,999.0
126057,Nissan,Sentra,2005,205000.0,East Granby,0,2.0,False,False,False,6026,999.0
151782,Mercedes-Benz,420-Class,1987,203189.0,New Windsor,0,2.0,False,False,False,12553,999.0
158302,Ford,Focus,2002,231000.0,Farmington,0,4.0,True,False,False,55024,999.0
158318,INFINITI,I30,1997,320058.0,Spanaway,0,6.0,False,False,False,98387,999.0
166047,Mazda,MAZDA6,2003,175000.0,Manchester,0,6.0,False,False,False,03103,999.0
326052,Ford,Taurus,1997,104537.0,Edison,0,2.0,False,False,False,8817,999.0
481932,Mercury,Grand Marquis,1999,177832.0,Mchenry,0,4.0,True,False,False,60051,999.0
527526,Kia,Spectra,2004,NaN,Peninsula,0,4.0,False,False,False,44264,999.0


In [69]:
df.loc[df["price"] == 999, "dealer_zip"].value_counts(dropna=False)

dealer_zip
32211    44
55024     7
55906     3
7608      2
56301     2
57401     2
84054     2
6026      1
12553     1
55024     1
98387     1
03103     1
8817      1
60051     1
44264     1
60636     1
22191     1
20748     1
49058     1
37601     1
30458     1
32714     1
34748     1
50401     1
33619     1
33771     1
33168     1
56258     1
68701     1
41042     1
56387     1
84070     1
84095     1
59601     1
84047     1
72764     1
66720     1
68901     1
89048     1
91786     1
Name: count, dtype: int64

In [70]:
df.loc[
    df["price"] < 999,
    [
        "make_name",
        "model_name",
        "year",
        "mileage",
        "city",
        "is_new",
        "owner_count",
        "has_accidents",
        "frame_damaged",
        "salvage",
        'dealer_zip',
        "price"
    ]
].sort_values("price").head(30)

,make_name,model_name,year,mileage,city,is_new,owner_count,has_accidents,frame_damaged,salvage,dealer_zip,price
1320599,Buick,Century,1999,190000.0,Belle Glade,0,4.0,False,False,False,33430,165.0
1359718,Ford,Explorer,2005,150000.0,Medley,0,4.0,False,False,False,33178,200.0
878365,Buick,Century,2005,202158.0,Traverse City,0,5.0,True,False,False,49684,249.0
1271923,Acura,RL,2006,NaN,Bainbridge,0,3.0,False,False,False,39817,250.0
1271913,Mitsubishi,Lancer,2008,NaN,Bainbridge,0,2.0,True,False,False,39817,250.0
1271928,Nissan,Altima Coupe,2008,NaN,Bainbridge,0,5.0,True,False,False,39817,250.0
1271918,Toyota,Avalon,2005,NaN,Bainbridge,0,5.0,True,False,False,39817,250.0
1271941,Kia,Sorento,2006,NaN,Bainbridge,0,6.0,True,False,False,39817,250.0
1938898,Mercury,Villager,1998,104000.0,Henryetta,0,3.0,False,False,False,74437,256.0
141899,Chrysler,Town & Country,2007,NaN,Mount Morris,0,1.0,False,False,False,48458,299.0


In [71]:
df.loc[df["price"] < 999, "dealer_zip"].value_counts(dropna=False)

dealer_zip
29073    83
60914    31
34266    11
33430    10
33935    10
         ..
81082     1
89801     1
80204     1
80920     1
81003     1
Name: count, Length: 168, dtype: int64

Target Price Checkpoint

- `price` has no missing, zero, or negative values.
- 494 listings are priced below $1,000.
- Repeated prices such as $999 appear across substantially different makes, models, years, mileage, and condition histories
- Many ultra-low prices are concentrated in specific dealer ZIP codes, suggesting placeholder or promotional pricing.

Current Conclusion

The repeated ultra-low prices across dissimilar vehicles, combined with their
concentration in specific dealer locations, suggest that some values below
$1,000 may represent dealer-specific pricing conventions, placeholder prices,
deposits, or other values that do not reflect full vehicle market value.

These records have not yet been removed from the original dataset. The current
proposed policy is to exclude listings priced below $1,000 from the dataset used
to train the initial current-value model while preserving them in the original
data.

In [77]:
valid_prices = df.loc[df['price'] >= 1000].copy()


In [87]:
valid_prices.shape

(2662757, 20)

In [85]:
valid_prices



,city,dealer_zip,engine_cylinders,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,vehicle_damage_category,wheel_system,year
0,Bayamon,960,I4,I4,NaN,Gasoline,NaN,177.0,1,Jeep,7.0,Renegade,NaN,23141.0,NaN,2.800000,A,Latitude FWD,NaN,FWD,2019
1,San Juan,922,I4,I4,NaN,Gasoline,NaN,246.0,1,Land Rover,8.0,Discovery Sport,NaN,46500.0,NaN,3.000000,A,S AWD,NaN,AWD,2020
2,Guaynabo,969,H4,H4,False,Gasoline,False,305.0,0,Subaru,NaN,WRX STI,3.0,46995.0,False,NaN,M,Base,NaN,AWD,2016
3,San Juan,922,V6,V6,NaN,Gasoline,NaN,340.0,1,Land Rover,11.0,Discovery,NaN,67430.0,NaN,3.000000,A,V6 HSE AWD,NaN,AWD,2020
4,San Juan,922,I4,I4,NaN,Gasoline,NaN,246.0,1,Land Rover,7.0,Discovery Sport,NaN,48880.0,NaN,3.000000,A,S AWD,NaN,AWD,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663246,Fairfield,94533,I4,I4,False,Gasoline,False,170.0,0,Chevrolet,41897.0,Equinox,1.0,17998.0,False,4.272727,A,1.5T LT FWD,NaN,FWD,2018
2663247,Vallejo,94591,V6,V6,NaN,Gasoline,NaN,310.0,1,Chevrolet,5.0,Traverse,NaN,36490.0,NaN,4.533333,A,LS FWD,NaN,FWD,2020
2663248,Napa,94559,NaN,NaN,False,NaN,True,240.0,0,Ford,57992.0,Fusion,2.0,12990.0,False,4.142857,A,SE,NaN,FWD,2016
2663249,Fairfield,94533,I4 Diesel,I4 Diesel,False,Diesel,False,180.0,0,Jaguar,27857.0,XE,1.0,26998.0,False,4.272727,A,20d Premium AWD,NaN,AWD,2017


In [80]:
valid_prices["price"].min()

np.float64(1000.0)

In [86]:
valid_prices.drop(columns=["vehicle_damage_category"], inplace=True)

In [88]:
valid_prices.drop_duplicates(inplace=True)

In [89]:
valid_prices.info()

<class 'pandas.DataFrame'>
Index: 2662757 entries, 0 to 2663250
Data columns (total 20 columns):
 #   Column            Dtype  
---  ------            -----  
 0   city              object 
 1   dealer_zip        object 
 2   engine_cylinders  object 
 3   engine_type       object 
 4   frame_damaged     object 
 5   fuel_type         object 
 6   has_accidents     object 
 7   horsepower        float64
 8   is_new            int64  
 9   make_name         object 
 10  mileage           float64
 11  model_name        object 
 12  owner_count       float64
 13  price             float64
 14  salvage           object 
 15  seller_rating     float64
 16  transmission      object 
 17  trim_name         object 
 18  wheel_system      object 
 19  year              int64  
dtypes: float64(5), int64(2), object(13)
memory usage: 426.6+ MB


In [92]:
valid_prices["engine_type"].equals (valid_prices["engine_cylinders"])

True

In [93]:
valid_prices.drop(columns =["engine_cylinders"], inplace = True)

In [94]:
valid_prices

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
0,Bayamon,960,I4,NaN,Gasoline,NaN,177.0,1,Jeep,7.0,Renegade,NaN,23141.0,NaN,2.800000,A,Latitude FWD,FWD,2019
1,San Juan,922,I4,NaN,Gasoline,NaN,246.0,1,Land Rover,8.0,Discovery Sport,NaN,46500.0,NaN,3.000000,A,S AWD,AWD,2020
2,Guaynabo,969,H4,False,Gasoline,False,305.0,0,Subaru,NaN,WRX STI,3.0,46995.0,False,NaN,M,Base,AWD,2016
3,San Juan,922,V6,NaN,Gasoline,NaN,340.0,1,Land Rover,11.0,Discovery,NaN,67430.0,NaN,3.000000,A,V6 HSE AWD,AWD,2020
4,San Juan,922,I4,NaN,Gasoline,NaN,246.0,1,Land Rover,7.0,Discovery Sport,NaN,48880.0,NaN,3.000000,A,S AWD,AWD,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663246,Fairfield,94533,I4,False,Gasoline,False,170.0,0,Chevrolet,41897.0,Equinox,1.0,17998.0,False,4.272727,A,1.5T LT FWD,FWD,2018
2663247,Vallejo,94591,V6,NaN,Gasoline,NaN,310.0,1,Chevrolet,5.0,Traverse,NaN,36490.0,NaN,4.533333,A,LS FWD,FWD,2020
2663248,Napa,94559,NaN,False,NaN,True,240.0,0,Ford,57992.0,Fusion,2.0,12990.0,False,4.142857,A,SE,FWD,2016
2663249,Fairfield,94533,I4 Diesel,False,Diesel,False,180.0,0,Jaguar,27857.0,XE,1.0,26998.0,False,4.272727,A,20d Premium AWD,AWD,2017
